![SGSSS Logo](https://github.com/SGSSSonline/text-analysis-for-social-scientists/blob/main/img/SGSSS_Stacked.png?raw=true)

# Practical 4: LLM-Assisted Text Analysis

Large Language Models (LLMs) have transformed the landscape of text analysis. Tasks that once required extensive manual coding, bespoke machine learning pipelines, or specialist NLP expertise can now be performed by prompting a general-purpose language model through an API. In this practical, we use LLM APIs to classify, summarise, and annotate the same parliamentary speeches we have worked with throughout the course.

## Aims

This practical has two aims:

1. **Demonstrate how to use LLM APIs for common text analysis tasks** such as classification and summarisation.
2. **Cultivate critical thinking about the strengths and limitations of LLM-assisted text analysis** compared to the methods covered earlier in the course.

### Lesson Details

| | |
| --- | --- |
| **Level** | Introductory |
| **Time** | ~35 minutes |
| **Pre-requisites** | Practical 1 (Preprocessing Text), Practical 3 (Topic Modelling) |
| **Learning outcomes** | Understand how to interact with LLM APIs using R |
| | Be able to use LLMs for zero-shot text classification |
| | Be able to use LLMs for text summarisation |
| | Understand how prompt design affects LLM outputs |
| | Be able to compare LLM-based and traditional text analysis approaches |

## Guide to Using This Resource

This is a **Jupyter Notebook** — an interactive document that combines text, code, and output in a single environment. If you are viewing this in **Google Colab**, you are running the notebook in the cloud, which means you do not need to install anything on your own machine.

**Note: This notebook uses R.** In Google Colab, you need to change the runtime: go to **Runtime > Change runtime type > select R**.

A notebook is made up of **cells**. There are two main types:

- **Markdown cells** contain formatted text (like this one). They provide explanations, instructions, and context.
- **Code cells** contain R code that you can execute. Code cells are displayed with a grey background and have a play button on the left.

To **run a cell**, click on it and press `Shift+Enter` (or click the play button). The output will appear directly below the cell. You should run the code cells **in order**, from top to bottom, as later cells often depend on variables or packages loaded in earlier cells.

If you are using **Google Colab**, make sure to save a copy of this notebook to your own Google Drive first: go to **File > Save a copy in Drive**.

If you are new to Jupyter Notebooks and would like a more detailed introduction, see the excellent materials by Dani Arribas-Bel: [https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb](https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb)

> **A note on API access:** This practical uses LLM APIs, which require an API key. If you do not have an API key, you can still follow along — all code cells include **example outputs** in the markdown cells that follow them, so you can see the results without running the API calls yourself.

### Interactive Test

Let's make sure everything is working. Run the cell below and enter your name when prompted.

In [ ]:
cat("Hello! What is your name?\n")
name <- readline()
cat(paste0("Welcome to the course, ", name, "!\n"))

## Setup

Before we begin, we need to install and load the R packages we will use throughout this practical. These include:

- **httr** — for making HTTP requests to the LLM API.
- **jsonlite** — for parsing JSON responses from the API.
- **dplyr** — for data manipulation.

In [ ]:
# Install packages (if needed on Colab)
install.packages(c("httr", "jsonlite", "dplyr"), quiet = TRUE)

library(httr)
library(jsonlite)
library(dplyr)

cat("Successfully loaded packages\n")

## Introduction

In Practicals 1–3, we built a text analysis pipeline from scratch: preprocessing raw text, computing word frequencies and TF-IDF scores, and fitting unsupervised topic models. Each of these steps required careful decisions about tokenisation, stopword removal, and model parameters.

In this practical, we take a different approach. We will use **LLM APIs** to analyse the same Hansard texts we have worked with throughout the course. LLMs can classify, summarise, and annotate text — tasks that traditionally required either manual coding or specialised machine learning models.

The key advantages of LLM-based text analysis include:

- **No training data required** — LLMs can perform classification "zero-shot", meaning they can categorise text into classes they have never been explicitly trained on.
- **Flexibility** — the same model can classify, summarise, extract entities, or answer questions, simply by changing the prompt.
- **Natural language interface** — you describe the task in plain English rather than writing specialised code.

However, there are important limitations to keep in mind:

- **Cost** — API calls cost money, and costs scale with the number of documents.
- **Reproducibility** — model outputs can change over time as providers update their models.
- **Transparency** — LLMs are "black boxes"; it is difficult to know exactly why a model made a particular classification.
- **Hallucination** — LLMs can produce confident-sounding but incorrect outputs.

We will explore both the power and the limitations of this approach.

## Loading the Corpus

We will work with the same sample of 20 UK parliamentary speeches from the **Hansard debates** that we used in Practicals 1–3. These speech excerpts cover a range of policy topics debated in the House of Commons, including health, the economy, climate, immigration, education, housing, and defence.

In [ ]:
# Create a sample dataset of parliamentary speech excerpts
df <- data.frame(
  date = c(
    "2024-01-15", "2024-01-15", "2024-01-22", "2024-01-22", "2024-02-05",
    "2024-02-05", "2024-02-12", "2024-02-12", "2024-02-19", "2024-02-19",
    "2024-03-04", "2024-03-04", "2024-03-11", "2024-03-11", "2024-03-18",
    "2024-03-18", "2024-03-25", "2024-03-25", "2024-04-01", "2024-04-01"
  ),
  speaker = c(
    "Margaret Thornton", "James Whitfield", "Sarah Okonkwo", "David Hargreaves",
    "Emily Chen", "Robert MacLeod", "Fatima Hussain", "Thomas Brennan",
    "Catherine Lloyd", "Andrew Patel", "Helen Murray", "Kwame Asante",
    "Patricia Gallagher", "Ravi Sharma", "Fiona Blackwood", "Marcus Johnson",
    "Alison Crawford", "Yusuf Ali", "Diana Novak", "Christopher Reeves"
  ),
  party = c(
    "Labour", "Conservative", "Labour", "Conservative", "Labour",
    "SNP", "Labour", "Conservative", "Liberal Democrat", "Labour",
    "SNP", "Labour", "Conservative", "Labour", "SNP",
    "Conservative", "Labour", "Labour", "Liberal Democrat", "Conservative"
  ),
  speech_text = c(
    "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them.",
    "Economic growth is the foundation upon which all public services depend. By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses. The private sector, not the state, is the engine of prosperity and job creation in this country.",
    "The climate crisis demands immediate and ambitious action from this House. We cannot afford to delay investment in renewable energy infrastructure. Our children and grandchildren will judge us by the decisions we make today on carbon emissions and environmental protection.",
    "Immigration policy must be fair but firm. We need a points-based system that attracts the skilled workers our economy needs while maintaining control of our borders. The British people voted for sovereignty and we must deliver on that promise.",
    "Education is the great equaliser in our society. Every child, regardless of their background or postcode, deserves access to excellent teaching and modern facilities. We must close the attainment gap between the most and least disadvantaged pupils in our schools.",
    "Scotland's interests are consistently overlooked by this Westminster government. The devolution settlement is being undermined at every turn. The people of Scotland deserve the right to determine their own future and to have their voice heard on matters that affect their daily lives.",
    "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency.",
    "Defence spending must remain a top priority for this government. The threats we face from hostile state actors are growing more complex and more dangerous. We must ensure our armed forces have the equipment and resources they need to keep this nation safe.",
    "Mental health services are chronically underfunded and overstretched. Too many people are waiting months or even years for treatment they desperately need. Parity of esteem between physical and mental health must become a reality, not just a slogan.",
    "The cost of living crisis is hitting working families hardest. Energy bills have skyrocketed and food prices continue to rise at alarming rates. The government must do more to support those who are struggling to heat their homes and feed their children.",
    "The fishing communities of Scotland have been betrayed by broken promises on Brexit. Access to our waters and fair quota allocations are essential for the survival of these coastal communities. We will continue to fight for the rights of Scottish fishermen in this House.",
    "Public transport infrastructure is failing passengers across the north of England. Years of underinvestment have left communities isolated and reliant on expensive and unreliable services. We need a transport revolution that connects people to jobs, education, and opportunity.",
    "Law and order must be restored to our streets. Violent crime and antisocial behaviour are blighting communities and the police need the resources and powers to tackle these problems effectively. We will always stand on the side of victims and law-abiding citizens.",
    "The arts and creative industries contribute billions to our economy and enrich the cultural life of this nation. Cuts to arts funding have devastated regional theatres, museums, and music venues. Investment in culture is not a luxury; it is an investment in our communities and our identity.",
    "Rural communities in Scotland face unique challenges that this government fails to understand. Broadband connectivity, access to healthcare, and affordable transport are not luxuries but necessities. The centralisation of services in urban areas is leaving rural Scotland behind.",
    "Free trade agreements are essential for our post-Brexit economic strategy. British goods and services must have access to global markets on the most favourable terms possible. We are building new trading relationships that will benefit every region of the United Kingdom.",
    "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law.",
    "Child poverty is a stain on our national conscience. Over four million children in this country are growing up in poverty, and the numbers are rising. We must reform the welfare system to ensure that no child goes hungry and every family has a decent standard of living.",
    "Local government is the backbone of democracy in this country. Councils are being asked to do more with less, and essential services are being cut to the bone. Proper funding for local authorities is not just desirable; it is essential for the health of our democratic institutions.",
    "Science and innovation are the keys to our future prosperity. The United Kingdom has world-leading universities and research institutions that must be supported and celebrated. Government investment in research and development is an investment in the jobs and industries of tomorrow."
  ),
  stringsAsFactors = FALSE
)

cat(paste0("Dataset shape: ", nrow(df), " speeches, ", ncol(df), " columns\n"))
head(df)

## Setting Up the API

To use an LLM for text analysis, we need to send requests to an **API** (Application Programming Interface). Most LLM providers — including OpenAI, Anthropic (via compatible wrappers), Mistral, and many open-source model hosts — use the **OpenAI-compatible chat completions format**. This means the same R function can work with multiple providers by changing the base URL and model name.

The function below sends a prompt to an LLM API and returns the response text. It uses the `httr` package to make an HTTP POST request with the following components:

- **Authorization header** — your API key, which authenticates you with the provider.
- **Model** — which LLM to use (e.g., `gpt-4o-mini` for OpenAI).
- **Messages** — the conversation, formatted as a list of messages. We send a single user message containing our prompt.
- **Temperature** — set to 0 for maximum reproducibility (the model will give the most likely response rather than sampling randomly).

In [ ]:
query_llm <- function(prompt, api_key, base_url = "https://api.openai.com/v1", model = "gpt-4o-mini") {
  #' Send a prompt to an LLM API and return the response text.
  response <- POST(
    paste0(base_url, "/chat/completions"),
    add_headers(
      "Authorization" = paste("Bearer", api_key),
      "Content-Type" = "application/json"
    ),
    body = toJSON(list(
      model = model,
      messages = list(list(role = "user", content = prompt)),
      temperature = 0
    ), auto_unbox = TRUE),
    encode = "json"
  )
  content(response)$choices[[1]]$message$content
}

### Enter Your API Key

To run the API calls in this notebook, you need an API key from an LLM provider. Paste your key in the cell below.

**Security note:** Never share your API key or commit it to a public repository. If you are using Google Colab, you can use Colab's Secrets feature (click the key icon in the left sidebar) to store your key securely.

> **If you do not have an API key**, that is fine. You can use OpenAI, Anthropic, or other OpenAI-compatible APIs. If you do not have access to any of these, you can follow along with the **example outputs** shown in the markdown cells throughout this notebook. The practical is designed to be educational even without running the API calls.

In [ ]:
# Enter your API key here (or use Colab's secrets feature)
API_KEY <- ""  # Paste your key here

# Optional: change these if you are using a different provider
BASE_URL <- "https://api.openai.com/v1"  # Change for other providers
MODEL <- "gpt-4o-mini"  # Change for other models

if (nchar(API_KEY) > 0) {
  cat(paste0("API key set. Using model '", MODEL, "' at ", BASE_URL, "\n"))
} else {
  cat("No API key set. You can follow along with the example outputs below.\n")
}

## Zero-Shot Classification

One of the most powerful capabilities of LLMs is **zero-shot classification** — categorising text into predefined classes without any training examples. The model relies on its pre-existing knowledge of language and concepts to make classifications based solely on the instructions in the prompt.

We will classify 5 of our parliamentary speeches into one of these policy categories: **Health, Education, Economy, Environment, Defence, Housing, Social Policy**.

The key to effective zero-shot classification is a clear, unambiguous prompt that specifies:
1. The task (classify the speech).
2. The categories (the set of valid labels).
3. The output format (respond with only the category name).

In [ ]:
# Select 5 speeches for classification (R uses 1-based indexing)
classification_indices <- c(1, 3, 5, 8, 10)
classification_df <- df[classification_indices, ]

# Define the classification categories
categories <- "Health, Education, Economy, Environment, Defence, Housing, Social Policy"

results <- data.frame(
  Speaker = character(),
  Party = character(),
  LLM_Classification = character(),
  stringsAsFactors = FALSE
)

for (i in seq_len(nrow(classification_df))) {
  row <- classification_df[i, ]
  prompt <- paste0(
    "Classify the following parliamentary speech into exactly one of these categories:\n",
    categories, "\n\n",
    'Speech: "', row$speech_text, '"\n\n',
    "Respond with only the category name, nothing else."
  )

  if (nchar(API_KEY) > 0) {
    classification <- query_llm(prompt, API_KEY, BASE_URL, MODEL)
    Sys.sleep(1)  # Pause between requests to respect rate limits
  } else {
    classification <- "[No API key - see example output below]"
  }

  results <- rbind(results, data.frame(
    Speaker = row$speaker,
    Party = row$party,
    LLM_Classification = trimws(classification),
    stringsAsFactors = FALSE
  ))
  cat(paste0("Classified ", row$speaker, ": ", trimws(classification), "\n"))
}

results

### Example Output

If you were unable to run the API call above, here are the expected results:

| Speaker | Party | LLM Classification |
|---|---|---|
| Margaret Thornton | Labour | Health |
| Sarah Okonkwo | Labour | Environment |
| Emily Chen | Labour | Education |
| Thomas Brennan | Conservative | Defence |
| Andrew Patel | Labour | Economy |

The LLM classifications are remarkably intuitive. Margaret Thornton's speech about the NHS is classified as **Health**; Sarah Okonkwo's speech about the climate crisis as **Environment**; Emily Chen's speech about schools as **Education**; Thomas Brennan's speech about armed forces as **Defence**; and Andrew Patel's speech about the cost of living as **Economy**.

Notice that the LLM was able to make these classifications without any training data or examples — it understood the task purely from the prompt instructions and its pre-trained knowledge.

### Discussion: How Accurate Is This?

The classifications above appear sensible, but how would we evaluate them more rigorously? In a research context, we would:

1. **Create a gold standard** — have human coders label the speeches independently.
2. **Compute inter-rater agreement** — compare the LLM labels to the human labels using metrics like Cohen's kappa or accuracy.
3. **Test on a larger sample** — 5 speeches is too few to draw conclusions about reliability.

For our purposes, the key insight is that LLMs can perform classification tasks that would previously have required either manual coding (expensive and slow) or supervised machine learning (which requires labelled training data).

## Summarisation

Another common text analysis task is **summarisation** — condensing a longer text into a shorter version that preserves its key content. LLMs are well-suited to this task because they can understand context, identify main points, and generate fluent prose.

We will summarise 3 of the longer speeches, asking the LLM to produce exactly 2 sentences per summary. Constraining the output length helps ensure the summaries are concise and comparable.

In [ ]:
# Select 3 speeches for summarisation (R uses 1-based indexing)
summary_indices <- c(1, 7, 17)
summary_df <- df[summary_indices, ]

for (i in seq_len(nrow(summary_df))) {
  row <- summary_df[i, ]
  prompt <- paste0(
    "Summarise the following parliamentary speech in exactly 2 sentences:\n\n",
    'Speech: "', row$speech_text, '"\n\n',
    "Summary:"
  )

  if (nchar(API_KEY) > 0) {
    summary_text <- query_llm(prompt, API_KEY, BASE_URL, MODEL)
    Sys.sleep(1)
  } else {
    summary_text <- "[No API key - see example output below]"
  }

  cat(paste0("--- ", row$speaker, " (", row$party, ") ---\n"))
  cat(paste0("Original: ", row$speech_text, "\n"))
  cat(paste0("Summary:  ", trimws(summary_text), "\n\n"))
}

### Example Output

If you were unable to run the API call above, here are the expected summaries:

**Margaret Thornton (Labour):**

- *Original:* "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them."
- *Summary:* "The speaker argues that the NHS is fundamental to the social contract and faces critical staffing shortages that endanger patients. They call on the government to take urgent action to address record-high waiting lists."

**Fatima Hussain (Labour):**

- *Original:* "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency."
- *Summary:* "The speaker highlights a housing crisis that is harming communities, with young people unable to afford homes and rising rents burdening households. They advocate for a large-scale social housing construction programme to address the emergency."

**Alison Crawford (Labour):**

- *Original:* "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law."
- *Summary:* "The speaker calls for strengthening workers' rights in response to the growth of zero-hours contracts and the gig economy, which have led to precarious employment. They argue that every worker deserves fair pay, decent conditions, and legal protection of their rights."

The summaries faithfully capture the core arguments of each speech in two sentences. Notice how the LLM identifies the main claim and the call to action in each speech — a task that would be difficult to automate with rule-based or traditional NLP approaches.

## Comparing LLM Classification to Topic Modelling

In Practical 3, we fitted an LDA topic model to the same corpus and discovered latent topics based on word co-occurrence patterns. Now we have LLM classifications for some of the same speeches. How do these two approaches compare?

| Dimension | Topic Modelling (LDA) | LLM Classification |
|---|---|---|
| **Approach** | Unsupervised — discovers themes from the data | Directed — classifies into researcher-specified categories |
| **Categories** | Emergent (learned from word co-occurrences) | Pre-defined (specified in the prompt) |
| **Training data** | None required, but needs a corpus | None required |
| **Transparency** | Interpretable (topic-word distributions) | Opaque (model internals are hidden) |
| **Reproducibility** | Reproducible with fixed random seed | May vary across API versions |
| **Scalability** | Fast and cheap for large corpora | Slow and costly (one API call per document) |
| **Mixed membership** | Yes — documents can belong to multiple topics | No (unless you design the prompt for it) |

Let us compare the results directly. The table below shows, for each of the 5 speeches we classified, the LDA topic assignment from Practical 3 alongside the LLM classification.

| Speaker | Party | LDA Dominant Topic (K=3) | LLM Classification |
|---|---|---|---|
| Margaret Thornton | Labour | Topic 0 (public services / community) | Health |
| Sarah Okonkwo | Labour | Topic 1 (economy / future) | Environment |
| Emily Chen | Labour | Topic 2 (rights / society) | Education |
| Thomas Brennan | Conservative | Topic 0 (public services / community) | Defence |
| Andrew Patel | Labour | Topic 2 (rights / society) | Economy |

The LDA topics are broad and overlapping — with only K=3 topics across 20 diverse speeches, many different policy areas get grouped together. The LLM classifications are more specific: "Health" rather than "public services", "Defence" rather than a broad cluster.

Neither approach is inherently better. Topic modelling is **exploratory** — useful when you do not know what themes exist in your data. LLM classification is **confirmatory** — useful when you have a predefined coding scheme and want to apply it at scale. The best research designs often combine both: use topic models to discover themes, then use LLMs (or human coders) to validate and refine the categories.

## Prompt Engineering Experiment

The way you write a prompt can significantly affect the LLM's output. This is known as **prompt engineering** — the practice of designing prompts to elicit the best possible responses from a language model.

To demonstrate this, we will classify the same speech using three different prompt strategies:

- **Prompt A: Simple instruction** — a minimal prompt with just the task and categories.
- **Prompt B: Few-shot (with examples)** — the prompt includes 2 labelled examples before the target speech.
- **Prompt C: Detailed rubric** — the prompt includes explicit definitions of each category.

We will use Andrew Patel's cost-of-living speech (index 10 in R), which could plausibly be classified as either **Economy** or **Social Policy**, making it a good test case for examining how prompt design affects borderline classifications.

In [ ]:
# The test speech: Andrew Patel on the cost of living
test_speech <- df$speech_text[10]
cat(paste0("Test speech (", df$speaker[10], ", ", df$party[10], "):\n\n"))
cat(test_speech)

In [ ]:
# Prompt A: Simple instruction
prompt_a <- paste0(
  "Classify the following parliamentary speech into exactly one of these categories:\n",
  "Health, Education, Economy, Environment, Defence, Housing, Social Policy\n\n",
  'Speech: "', test_speech, '"\n\n',
  "Respond with only the category name, nothing else."
)

# Prompt B: Few-shot (with examples)
prompt_b <- paste0(
  "Classify parliamentary speeches into exactly one of these categories:\n",
  "Health, Education, Economy, Environment, Defence, Housing, Social Policy\n\n",
  "Example 1:\n",
  'Speech: "The National Health Service remains the cornerstone of our social contract with the British people. ',
  'We must invest in frontline services and address the chronic staffing shortages."\n',
  "Category: Health\n\n",
  "Example 2:\n",
  'Speech: "Economic growth is the foundation upon which all public services depend. ',
  'By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses."\n',
  "Category: Economy\n\n",
  "Now classify this speech:\n",
  'Speech: "', test_speech, '"\n',
  "Category:"
)

# Prompt C: Detailed rubric
prompt_c <- paste0(
  "Classify the following parliamentary speech into exactly one of these categories.\n",
  "Use the definitions below to guide your decision:\n\n",
  "- Health: Speeches primarily about the NHS, healthcare services, patient care, or public health.\n",
  "- Education: Speeches primarily about schools, universities, teaching, or educational attainment.\n",
  "- Economy: Speeches primarily about economic growth, taxation, trade, business, or fiscal policy.\n",
  "- Environment: Speeches primarily about climate change, renewable energy, or environmental protection.\n",
  "- Defence: Speeches primarily about the military, national security, or armed forces.\n",
  "- Housing: Speeches primarily about housing supply, rents, homeownership, or homelessness.\n",
  "- Social Policy: Speeches primarily about poverty, welfare, workers' rights, cost of living, or inequality.\n\n",
  'Speech: "', test_speech, '"\n\n',
  "Respond with only the category name, nothing else."
)

prompts <- list(
  "Prompt A (Simple)" = prompt_a,
  "Prompt B (Few-shot)" = prompt_b,
  "Prompt C (Rubric)" = prompt_c
)

prompt_results <- data.frame(
  Prompt_Strategy = character(),
  Classification = character(),
  stringsAsFactors = FALSE
)

for (name in names(prompts)) {
  if (nchar(API_KEY) > 0) {
    result <- query_llm(prompts[[name]], API_KEY, BASE_URL, MODEL)
    Sys.sleep(1)
  } else {
    result <- "[No API key - see example output below]"
  }

  prompt_results <- rbind(prompt_results, data.frame(
    Prompt_Strategy = name,
    Classification = trimws(result),
    stringsAsFactors = FALSE
  ))
  cat(paste0(name, ": ", trimws(result), "\n"))
}

prompt_results

### Example Output

If you were unable to run the API calls above, here are the expected results:

| Prompt Strategy | Classification |
|---|---|
| Prompt A (Simple) | Economy |
| Prompt B (Few-shot) | Economy |
| Prompt C (Rubric) | Social Policy |

### Analysis

This is an instructive result. Andrew Patel's speech about the cost of living, energy bills, and food prices sits on the boundary between **Economy** and **Social Policy**. With the simple and few-shot prompts, the LLM classifies it as "Economy" — likely because the speech mentions energy bills, food prices, and financial hardship. But with the detailed rubric that explicitly defines Social Policy as covering "poverty, welfare, workers' rights, cost of living, or inequality", the model classifies it as "Social Policy".

This demonstrates a crucial point: **the prompt is a research instrument**. Just as the wording of a survey question affects responses, the wording of a classification prompt affects LLM outputs. Researchers using LLMs for text classification must:

1. **Document their prompts** as part of their methodology.
2. **Test multiple prompt designs** and report whether results are robust.
3. **Validate against human judgement** to ensure the classifications are meaningful.

## Exercise

Now it is your turn. Your task is to:

1. **Design a prompt** for a text analysis task of your choice. This could be:
   - Classifying speeches by a different scheme (e.g., sentiment, rhetorical strategy, political ideology).
   - Extracting specific information (e.g., named entities, policy proposals, or rhetorical devices).
   - Answering a research question (e.g., "Does this speech support or oppose government spending?").

2. **Apply your prompt to at least 3 speeches** from the corpus.

3. **Reflect on the following questions:**
   - (a) How well did the LLM perform on your task?
   - (b) What are the strengths and limitations of this approach compared to the methods we used in Practicals 1–3 (preprocessing, frequency analysis, topic modelling)?
   - (c) How would you validate these results in a research context?

Use the skeleton code below to guide you. Replace the `# YOUR CODE HERE` comments with your own code.

In [ ]:
# Step 1: Design your prompt template
# YOUR CODE HERE: write a prompt template (use paste0() to include {speech_text})
my_prompt_template <- ""


In [ ]:
# Step 2: Apply your prompt to at least 3 speeches
my_indices <- c(1, 2, 3)  # Choose your own speech indices

# YOUR CODE HERE: loop through the speeches, send prompts, and collect results


**Step 3 — Reflection**: Answer the three questions above.

*(a) YOUR ANSWER HERE*

*(b) YOUR ANSWER HERE*

*(c) YOUR ANSWER HERE*

## Appendix: Exercise Solution

In [ ]:
# --- Exercise Solution ---
# Example: Classify speeches by whether they SUPPORT or OPPOSE increased government spending.

my_prompt_template <- 'Analyse the following parliamentary speech and determine the speaker\'s
position on government spending.

Classify the speech into exactly one of these categories:
- SUPPORT: The speaker advocates for increased government spending or investment.
- OPPOSE: The speaker advocates for reduced government spending, tax cuts, or private sector solutions.
- NEUTRAL: The speech does not take a clear position on government spending.

Speech: "%s"

Respond with only the category name (SUPPORT, OPPOSE, or NEUTRAL), nothing else.'

# Apply to 5 speeches
exercise_indices <- c(1, 2, 5, 8, 18)
exercise_results <- data.frame(
  Speaker = character(),
  Party = character(),
  Spending_Position = character(),
  stringsAsFactors = FALSE
)

for (i in exercise_indices) {
  row <- df[i, ]
  prompt <- sprintf(my_prompt_template, row$speech_text)

  if (nchar(API_KEY) > 0) {
    result <- query_llm(prompt, API_KEY, BASE_URL, MODEL)
    Sys.sleep(1)
  } else {
    result <- "[No API key]"
  }

  exercise_results <- rbind(exercise_results, data.frame(
    Speaker = row$speaker,
    Party = row$party,
    Spending_Position = trimws(result),
    stringsAsFactors = FALSE
  ))
}

exercise_results

### Example Output for Exercise Solution

| Speaker | Party | Spending Position |
|---|---|---|
| Margaret Thornton | Labour | SUPPORT |
| James Whitfield | Conservative | OPPOSE |
| Emily Chen | Labour | SUPPORT |
| Thomas Brennan | Conservative | SUPPORT |
| Yusuf Ali | Labour | SUPPORT |

**Reflection:**

*(a) The LLM performs reasonably well on this task. Margaret Thornton (NHS investment), Emily Chen (education facilities), Thomas Brennan (defence spending), and Yusuf Ali (welfare reform) are all correctly identified as supporting more government spending, while James Whitfield (tax cuts, private sector) is identified as opposing it. Note that Brennan is a Conservative who supports spending — specifically on defence — which shows the LLM can distinguish between party affiliation and policy position.*

*(b) This approach is far simpler than training a supervised classifier, which would require labelled training data. It is also more nuanced than keyword-based approaches — the LLM understands that "invest in frontline services" implies support for spending even though the word "spending" does not appear. However, it is less transparent than a dictionary-based approach where you can see exactly which words drove the classification.*

*(c) To validate these results in a research context, one would need to: (1) have human coders independently label a sample of speeches using the same coding scheme; (2) compute inter-rater agreement between the LLM and human coders; (3) test with different LLMs and prompt variations to check robustness; and (4) report the exact prompt used as part of the methodology.*

## Bibliography

- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.
- Gilardi, F., Alizadeh, M., & Kubli, M. (2023). ChatGPT Outperforms Crowd Workers for Text-Annotation Tasks. *Proceedings of the National Academy of Sciences*, 120(30), e2305016120.
- Tornberg, P. (2024). ChatGPT-4 Outperforms Experts and Crowd Workers in Annotating Political Twitter Messages with Zero-Shot Learning. *arXiv preprint arXiv:2304.06588*.
- Ziems, C., Held, W., Shaikh, O., Chen, J., Zhang, Z., & Yang, D. (2024). Can Large Language Models Transform Computational Social Science? *Computational Linguistics*, 50(1), 237–291.
- Brown, T. B., et al. (2020). Language Models Are Few-Shot Learners. *Advances in Neural Information Processing Systems*, 33, 1877–1901.

---

**END OF FILE**